# MS MARCO MiniLM L4 Reranker

**Model:** cross-encoder/ms-marco-MiniLM-L-4-v2 | **Size:** 23MB | **Product:** prod-g6jlxeruthwzs

The fastest reranker in the MS MARCO MiniLM family. A cross-encoder trained on the MS MARCO passage ranking dataset, designed to rerank retrieval results for improved relevance. At just 23MB, it is the go-to choice for high-throughput reranking with minimal latency.

## Use Cases
- Second-stage reranking in RAG (Retrieval-Augmented Generation) pipelines
- Search result relevance improvement
- Question answering passage selection
- E-commerce product search reranking

In [ ]:
import boto3
import sagemaker
from sagemaker import ModelPackage

region = boto3.Session().region_name
role = sagemaker.get_execution_role()
sm_client = boto3.client('sagemaker', region_name=region)

print(f'Region: {region}')
print(f'Role: {role}')

In [ ]:
# Replace with your actual Model Package ARN from AWS Marketplace
model_package_arn = 'arn:aws:sagemaker:REGION:ACCOUNT:model-package/MODEL_PACKAGE_NAME'

# Validate ARN before deploying
if 'REGION' in model_package_arn or 'ACCOUNT' in model_package_arn or 'MODEL_PACKAGE_NAME' in model_package_arn:
    raise ValueError(
        'model_package_arn contains placeholder values. '
        'Subscribe to the model on AWS Marketplace and replace with the actual ARN.'
    )

endpoint_name = 'ms-marco-minilm-l4-reranker'
instance_type = 'ml.m5.xlarge'

try:
    model = ModelPackage(
        role=role,
        model_package_arn=model_package_arn,
        sagemaker_session=sagemaker.Session()
    )
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )
    print(f'Endpoint deployed: {endpoint_name}')
except Exception as e:
    print(f'Deployment failed: {e}')
    raise

## Step 2: Run Inference

Send a query and a list of passages to rerank. The model returns relevance scores for each query-passage pair.

In [ ]:
import json

runtime = boto3.client('sagemaker-runtime', region_name=region)

query = 'How do transformers work in natural language processing?'
passages = [
    'Transformers use self-attention mechanisms to process sequences in parallel, enabling efficient training on large corpora.',
    'The Eiffel Tower is located in Paris and was built in 1889.',
    'BERT and GPT are both transformer-based models used for NLP tasks like classification and generation.',
]

payload = json.dumps({'query': query, 'passages': passages})

try:
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=payload
    )
    result_raw = response['Body'].read().decode('utf-8')
    try:
        result = json.loads(result_raw)
        print('Reranking scores:', result)
    except json.JSONDecodeError:
        print('Raw response:', result_raw)
except Exception as e:
    print(f'Inference failed: {e}')
    raise

In [ ]:
# Cleanup - delete the endpoint to avoid ongoing charges
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f'Endpoint {endpoint_name} deleted.')
except Exception as e:
    print(f'Cleanup failed: {e}')